# MoE Routing Analysis

Visualises how the Mixture-of-Experts router assigns tokens to experts.
Three perspectives:

1. **Expert Activation Heatmap** — which experts get used, and by which layers?
2. **Load Uniformity** — are tokens evenly distributed across experts?
3. **Routing Entropy** — how confident is the router in its decisions?

$$
\begin{aligned}
\text{Router:}\quad & \text{logits} = x W_{\text{gate}}^T \in \mathbb{R}^{n_{\text{experts}}} \\
& \text{indices} = \text{topk}(\text{logits}, k), \quad
\text{weights} = \text{softmax}(\text{logits}[\text{indices}]) \\
& H_{\text{routing}} = -\sum_{i=1}^k w_i \log w_i \quad
\text{(routing entropy, max = }\ln k \text{ for uniform)}
\end{aligned}
$$

---
## 1. Setup

Load the project modules and (optionally) a trained MoE checkpoint.

In [ ]:
import sys
from pathlib import Path

project_root = str(Path.cwd().parent) if Path.cwd().name == "apps" else str(Path.cwd())
if project_root not in sys.path:
    sys.path.append(project_root)

import math
from collections import defaultdict

import matplotlib.pyplot as plt
import torch
from core.transformer import GPT, BPETokenizer
from core.transformer.checkpoint import load_checkpoint
from core.transformer.moe import MoEFFN
from matplotlib.colors import Normalize

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {DEVICE}")

In [ ]:
# ── Load trained MoE model, or create a small untrained demo ───
MODEL_DIR = Path(project_root) / "models"

if (MODEL_DIR / "training_config.pt").exists():
    model, tokenizer, config = load_checkpoint(MODEL_DIR)
    print("=== Training Config ===")
    for k, v in config.items():
        print(f"  {k}: {v}")
    print(f"\nBPE tokenizer: vocab_size={tokenizer.vocab_size}")
    print("Loaded trained checkpoint ✓")
    VOCAB_SIZE = config["vocab_size"]
    N_EXPERTS = config.get("n_experts", 8)
else:
    print("No trained model found — creating an untrained MoE demo model.")
    print("(Routing mechanics will be visible, but patterns reflect random weights)")
    tokenizer = BPETokenizer(vocab_size=256)
    tokenizer.train(["A quick brown fox jumps over the lazy dog."])
    model = GPT(
        vocab_size=256,
        d_model=64,
        n_layers=3,
        n_heads=4,
        d_ff=256,
        use_moe=True,
        n_experts=8,
        moe_k=2,
    )
    VOCAB_SIZE = 256
    N_EXPERTS = 8

model.eval()
model.to(DEVICE)
print(f"\nModel: {model}")
print(f"MoE enabled: {model.use_moe}")
print(f"N experts: {N_EXPERTS}")

In [ ]:
# ── Input text: sample from WikiText-2 (pure English, no CJK) ──
WIKITEXT_SAMPLE = """The Turing test was proposed by British mathematician Alan Turing in his 1950 paper Computing Machinery and Intelligence , which opens with the words : " I propose to consider the question , ' Can machines think ? ' " The term ' Artificial Intelligence ' was created at a conference held at Dartmouth College in 1956 . Allen Newell , J. C. Shaw , and Herbert A. Simon pioneered the newly created artificial intelligence field with the Logic Theory Machine ( 1956 ) , and the General Problem Solver in 1957 . In 1958 , John McCarthy and Marvin Minsky started the MIT Artificial Intelligence lab with $ 50 @,@ 000 . John McCarthy also created LISP in the summer of 1958 , a programming language still important in artificial intelligence research . """

# Determine max input from config if available, otherwise use 128
if (MODEL_DIR / "training_config.pt").exists():
    max_input = config["block_size"]
else:
    max_input = 128

ids = tokenizer.encode(WIKITEXT_SAMPLE)[:max_input]
token_ids = torch.tensor([ids], dtype=torch.long, device=DEVICE)
input_text = tokenizer.decode(ids, skip_special_tokens=True)

print(f"Input: {len(ids)} tokens")
print(f"First 160 chars:\n  {input_text[:160]}...")

---
## 2. Capturing Routing Decisions

We register **forward hooks** on each MoE block's router to intercept
the (weights, indices) output during a forward pass.

This gives us per-layer, per-token routing data without modifying
the core module code.

In [ ]:
# ── Hook: capture router output for every MoE layer ──
routing_data = {}


def make_hook(layer_idx):
    """Return a forward hook that stores router output."""

    def hook(module, inp, out):
        weights, indices, logits = out
        routing_data[layer_idx] = {
            "weights": weights.detach().cpu(),  # (batch, seq, k)
            "indices": indices.detach().cpu(),  # (batch, seq, k)
            "logits": logits.detach().cpu(),
        }

    return hook


handles = []
n_moe_layers = 0
for i, block in enumerate(model.blocks):
    if isinstance(block.ff, MoEFFN):
        h = block.ff.router.register_forward_hook(make_hook(i))
        handles.append(h)
        n_moe_layers += 1

print(f"Registered hooks on {n_moe_layers} MoE layer(s)")

# ── Run one forward pass ──
with torch.no_grad():
    logits = model(token_ids)

# ── Remove hooks ──
for h in handles:
    h.remove()

print(
    f"Captured routing data for {len(routing_data)} layer(s): {sorted(routing_data.keys())}"
)

# Examine one layer
sample_layer = sorted(routing_data.keys())[0]
d = routing_data[sample_layer]
print(f"\nLayer {sample_layer} routing data shapes:")
print(f"  weights  {tuple(d['weights'].shape)}  — (batch, seq, k)")
print(f"  indices  {tuple(d['indices'].shape)}  — (batch, seq, k)")
print(f"  logits   {tuple(d['logits'].shape)}  — (batch, seq, n_experts)")

# Quick check: expert assignment distribution
n_experts_actual = d["logits"].size(-1)
indices_flat = d["indices"].reshape(-1)
counts = torch.bincount(indices_flat, minlength=n_experts_actual)
print("\nExpert assignments (all layers combined):")
for e in range(n_experts_actual):
    print(
        f"  Expert {e}: {counts[e].item():4d} slots ({100 * counts[e].item() / indices_flat.size(0):.1f}%)"
    )

---
## 3. Expert Activation Heatmap

For each MoE layer, how many routing slots were assigned to each expert?
A uniform router would show equal counts across all experts.

The heatmap rows are layers and columns are experts.
Darker = more slots assigned to that (layer, expert) pair.

In [ ]:
sorted_layers = sorted(routing_data.keys())
n_experts = routing_data[sorted_layers[0]]["logits"].size(-1)

# Build the activation matrix: layers x experts
activation_matrix = torch.zeros(len(sorted_layers), n_experts)
for row, layer_idx in enumerate(sorted_layers):
    d = routing_data[layer_idx]
    flat_idx = d["indices"].reshape(-1)
    for e in range(n_experts):
        activation_matrix[row, e] = (flat_idx == e).sum().item()

# Normalise each row as a fraction of total slots
total_per_row = activation_matrix.sum(dim=1, keepdim=True)
activation_frac = activation_matrix / total_per_row.clamp(min=1)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# (a) Raw counts
im0 = axes[0].imshow(
    activation_matrix.numpy(),
    aspect="auto",
    cmap="YlOrRd",
)
axes[0].set_xlabel("Expert ID")
axes[0].set_ylabel("Layer")
axes[0].set_title("Expert Activation (raw counts)")
axes[0].set_xticks(range(n_experts))
axes[0].set_yticks(range(len(sorted_layers)))
axes[0].set_yticklabels([f"L{l}" for l in sorted_layers])
for i in range(len(sorted_layers)):
    for j in range(n_experts):
        val = int(activation_matrix[i, j].item())
        axes[0].text(j, i, str(val), ha="center", va="center", fontsize=7)
fig.colorbar(im0, ax=axes[0], label="token slots")

# (b) Fraction per row
uniform_line = 1.0 / n_experts
im1 = axes[1].imshow(
    activation_frac.numpy(),
    aspect="auto",
    cmap="YlOrRd",
    norm=Normalize(vmin=0, vmax=max(2 * uniform_line, activation_frac.max().item())),
)
axes[1].set_xlabel("Expert ID")
axes[1].set_ylabel("Layer")
axes[1].set_title(
    f"Expert Activation (fraction of slots)\n"
    f"Uniform = {uniform_line:.2f}, "
    f"actual range = [{activation_frac.min().item():.3f}, {activation_frac.max().item():.3f}]"
)
axes[1].set_xticks(range(n_experts))
axes[1].set_yticks(range(len(sorted_layers)))
axes[1].set_yticklabels([f"L{l}" for l in sorted_layers])
for i in range(len(sorted_layers)):
    for j in range(n_experts):
        val = activation_frac[i, j].item()
        axes[1].text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im1, ax=axes[1], label="fraction")

plt.tight_layout()
plt.show()

print(f"Expected uniform fraction per expert: {uniform_line:.3f} ({1}/{n_experts})")
print(
    f"Load imbalance score (CV = std/mean, 0=perfect): "
    f"{activation_frac.std().item() / activation_frac.mean().item():.4f}"
)

---
## 4. Expert Co-activation Matrix

With k=2 routing, each token selects **two** experts. This section shows
which expert pairs co-occur frequently, revealing complementarity:

- **Frequent pairs** → experts complement each other (both useful for the same token)
- **Rare/never pairs** → experts compete (router picks one *or* the other)

The diagonal is each expert co-occurring with itself — which should be zero
since top-k picks k distinct experts.

In [ ]:
# ── Build co-activation matrix: n_experts x n_experts ──
co_activation = torch.zeros(n_experts, n_experts, dtype=torch.long)

for layer_idx in sorted_layers:
    idx = routing_data[layer_idx]["indices"][0]  # (seq, k)
    for pos in range(idx.size(0)):
        e0, e1 = idx[pos, 0].item(), idx[pos, 1].item()
        co_activation[e0, e1] += 1
        co_activation[e1, e0] += 1

# Row-normalize: given expert i, fraction of co-occurrence with j
expert_total = co_activation.sum(dim=1, keepdim=True).float().clamp(min=1)
co_activation_norm = co_activation.float() / expert_total

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) Raw counts
im0 = axes[0].imshow(co_activation.numpy(), cmap="Blues", aspect="equal")
axes[0].set_title("Raw co-occurrence count")
axes[0].set_xlabel("Expert")
axes[0].set_ylabel("Expert")
for i in range(n_experts):
    for j in range(n_experts):
        val = int(co_activation[i, j].item())
        if val > 0:
            axes[0].text(j, i, str(val), ha="center", va="center", fontsize=7)
fig.colorbar(im0, ax=axes[0])

# (b) Row-normalized fraction
im1 = axes[1].imshow(
    co_activation_norm.numpy(), cmap="Blues", aspect="equal", vmin=0, vmax=1
)
axes[1].set_title("Co-occurrence fraction (row-normalized)")
axes[1].set_xlabel("Expert")
axes[1].set_ylabel("Expert")
for i in range(n_experts):
    for j in range(n_experts):
        axes[1].text(
            j,
            i,
            f"{co_activation_norm[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=7,
        )
fig.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

# ── Top pairs ──
all_pairs = []
for i in range(n_experts):
    for j in range(i + 1, n_experts):
        all_pairs.append((i, j, co_activation[i, j].item()))
all_pairs.sort(key=lambda x: -x[2])
print("Most frequent expert pairs:")
for i, j, c in all_pairs[:5]:
    print(f"  Expert {i} + Expert {j}: {int(c)} co-occurrences")
never = [(i, j) for i, j, c in all_pairs if c == 0]
if never:
    print(f"\nNever-co-occurring pairs ({len(never)}):")
    for i, j in never:
        print(f"  Expert {i} + Expert {j}")

---
## 5. Routing Entropy

For each token, the router outputs a 2-element weight vector (k=2).
The **routing entropy** measures how confident the router is:

$$H_{\text{route}} = -\sum_{i=1}^k w_i \log w_i$$

| Entropy | Meaning |
|---------|---------|
| $H \approx 0$ | Router is very confident — one expert dominates |
| $H \approx \ln 2$ | Router is uncertain — both experts equally weighted |

Low entropy is generally desirable: the router should specialise.

In [ ]:
H_MAX = math.log(2)  # max entropy for k=2

# Use sorted_layers instead of deleted n_layers variable
n_layers = len(sorted_layers)
n_cols = min(4, n_layers)
n_rows = (n_layers + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows))
axes = axes.flatten() if n_layers > 1 else [axes]

all_entropies = {}

for idx, layer_idx in enumerate(sorted_layers):
    ax = axes[idx]

    d = routing_data[layer_idx]
    w = d["weights"]  # (batch, seq, k)
    H = -(w * torch.log(w.clamp(min=1e-10))).sum(dim=-1)  # (batch, seq)
    all_entropies[layer_idx] = H

    # Histogram of per-token entropies
    ax.hist(
        H.flatten().numpy(), bins=30, alpha=0.7, color="steelblue", edgecolor="white"
    )
    ax.axvline(
        x=H_MAX,
        color="red",
        linestyle="--",
        alpha=0.6,
        label=f"max (uniform) = {H_MAX:.3f}",
    )
    ax.axvline(
        x=H.mean().item(),
        color="darkgreen",
        linestyle="--",
        alpha=0.6,
        label=f"mean = {H.mean().item():.3f}",
    )
    ax.set_title(f"Layer {layer_idx} — routing entropy")
    ax.set_xlabel("Entropy (nats)")
    ax.set_ylabel("Token count")
    ax.legend(fontsize=8)

# Hide unused axes
for idx in range(n_layers, len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()

# Summary table
print(
    f"{'Layer':>6} | {'Mean H':>8} | {'Std H':>7} | {'% Low (H<0.1)':>14} | {'% High (H>0.6)':>15}"
)
print("-" * 60)
for layer_idx in sorted_layers:
    H = all_entropies[layer_idx]
    H_flat = H.flatten()
    low_pct = (H_flat < 0.1).sum().item() / H_flat.numel() * 100
    high_pct = (H_flat > 0.6).sum().item() / H_flat.numel() * 100
    print(
        f"L{layer_idx:>4} | {H.mean().item():>8.4f} | {H.std().item():>7.4f} | {low_pct:>13.1f}% | {high_pct:>14.1f}%"
    )

In [ ]:
# ── Layer-wise entropy trend ──
# Shows how routing confidence evolves across layers
layer_means = []
layer_stds = []
for layer_idx in sorted_layers:
    H = all_entropies[layer_idx]
    layer_means.append(H.mean().item())
    layer_stds.append(H.std().item())

fig, ax = plt.subplots(figsize=(8, 4))

ax.errorbar(
    range(len(sorted_layers)),
    layer_means,
    yerr=layer_stds,
    fmt="-o",
    color="steelblue",
    capsize=5,
    capthick=2,
    markersize=8,
    linewidth=2,
)
ax.axhline(
    y=H_MAX, color="red", linestyle="--", alpha=0.5, label=f"Max entropy ({H_MAX:.3f})"
)
ax.axhline(y=0, color="gray", linestyle=":", alpha=0.5)

# Fill between std
ax.fill_between(
    range(len(sorted_layers)),
    [m - s for m, s in zip(layer_means, layer_stds)],
    [m + s for m, s in zip(layer_means, layer_stds)],
    alpha=0.15,
    color="steelblue",
)

ax.set_xticks(range(len(sorted_layers)))
ax.set_xticklabels([f"Layer {l}" for l in sorted_layers])
ax.set_ylabel("Mean routing entropy (nats)")
ax.set_title("Routing Entropy Across Layers")
ax.legend(fontsize=9)
ax.set_ylim(-0.05, H_MAX * 1.15)

# Annotate
for i, (m, s) in enumerate(zip(layer_means, layer_stds)):
    ax.annotate(
        f"{m:.3f}\n±{s:.3f}",
        (i, m),
        textcoords="offset points",
        xytext=(0, 15),
        ha="center",
        fontsize=8,
        arrowprops=dict(arrowstyle="->", color="gray", lw=0.5),
    )

plt.tight_layout()
plt.show()

# ── Interpretation ──
print(f"Layer-wise entropy range: [{min(layer_means):.4f}, {max(layer_means):.4f}]")
print()
print("Interpretation:")
print("  Low entropy  → router is confident (one expert dominates)")
print("  High entropy → router is uncertain (both experts ~equal weight)")

---
## 6. Per-Token Routing Path

Which experts handle each token in the sequence, across layers?
Each token is routed to k=2 experts per layer, visualised as colored squares.

This helps identify whether certain experts specialise in:
- Specific token types (punctuation, letters, spaces)
- Specific positions (early vs. late in the sequence)

In [ ]:
from matplotlib.lines import Line2D

n_layers_path = len(sorted_layers)
token_ids_np = token_ids[0].tolist()
token_chars = [tokenizer.id_to_token.get(t, "?") for t in token_ids_np]
seq_len = len(token_chars)

fig, axes = plt.subplots(n_layers_path, 1, figsize=(18, 3 * n_layers_path), sharex=True)
if n_layers_path == 1:
    axes = [axes]

for idx, layer_idx in enumerate(sorted_layers):
    ax = axes[idx]
    d = routing_data[layer_idx]
    indices = d["indices"][0]  # (seq, k)
    weights = d["weights"][0]  # (seq, k)

    for pos in range(min(seq_len, indices.size(0))):
        for slot in range(indices.size(-1)):
            expert = indices[pos, slot].item()
            w = weights[pos, slot].item()
            y_offset = slot * 0.9
            ax.scatter(
                pos,
                expert + y_offset,
                s=200 * w + 20,
                c=[[w]],  # 2D array for cmap to work
                cmap="viridis",
                vmin=0.3,
                vmax=0.7,
                alpha=0.8,
                edgecolors="black",
                linewidth=0.5,
            )

    ax.set_ylabel("Expert")
    ax.set_title(f"Layer {layer_idx} — routing path")
    ax.set_yticks(range(n_experts))
    ax.set_ylim(-0.5, n_experts - 0.5)

    # Legend using proxy artists
    if idx == 0:
        legend_elems = [
            Line2D(
                [0],
                [0],
                marker="o",
                color="w",
                markerfacecolor="#4dac26",
                markersize=6,
                label="w=0.4",
            ),
            Line2D(
                [0],
                [0],
                marker="o",
                color="w",
                markerfacecolor="#f7a81b",
                markersize=9,
                label="w=0.6",
            ),
            Line2D(
                [0],
                [0],
                marker="o",
                color="w",
                markerfacecolor="#d7191c",
                markersize=14,
                label="w≈1.0",
            ),
        ]
        ax.legend(
            handles=legend_elems,
            fontsize=7,
            title="routing weight",
            title_fontsize=8,
            loc="upper right",
        )

    if idx == n_layers_path - 1:
        ax.set_xticks(range(seq_len))
        ax.set_xticklabels(token_chars, fontsize=6, rotation=90)
        ax.set_xlabel("Token position")

plt.tight_layout()
plt.show()

---
## 7. Top BPE Tokens per Expert

Instead of raw characters (noisy with BPE), we show the **actual subword tokens**
that each expert prefers. This reveals whether experts learn semantic
specialisation — e.g., one expert handles punctuation, another handles
common words like "the", a third handles rare tokens.

For a selected layer, we group tokens by their assigned expert and show
the most frequent tokens per expert.

In [ ]:
from collections import Counter

# For a selected layer, show which BPE tokens each expert prefers
demo_layer = sorted_layers[0]
d = routing_data[demo_layer]
indices = d["indices"][0]  # (seq, k)

# Get the decoded text of each BPE token
# (decode directly via tokenizer)
token_ids_flat = token_ids[0].tolist()  # (seq,)
token_texts = [tokenizer.decode([t], skip_special_tokens=True) for t in token_ids_flat]

# Build: expert -> list of token strings
expert_tokens: dict[int, list[str]] = defaultdict(list)
for pos in range(indices.size(0)):
    for slot in range(indices.size(-1)):
        e = indices[pos, slot].item()
        expert_tokens[e].append(token_texts[pos])

# Display top tokens per expert as horizontal bar charts
n_cols = 4
n_rows = (n_experts + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 3 * n_rows))
axes = axes.flatten()

for e in range(n_experts):
    ax = axes[e]
    tokens_list = expert_tokens[e]
    if not tokens_list:
        ax.text(
            0.5,
            0.5,
            "(no tokens)",
            ha="center",
            va="center",
            transform=ax.transAxes,
            fontsize=10,
        )
        ax.set_title(f"Expert {e}")
        continue

    counter = Counter(tokens_list)
    top = counter.most_common(10)
    labels, counts = zip(*top) if top else ([], [])
    total = len(tokens_list)
    pcts = [c / total * 100 for c in counts]

    ax.barh(range(len(labels)), pcts, color="steelblue", edgecolor="k", linewidth=0.5)
    ax.set_yticks(range(len(labels)))
    display_labels = []
    for l in labels:
        if l == "\n":
            display_labels.append(r"\n")
        elif l == " ":
            display_labels.append(r"␣")
        elif not l.isprintable():
            display_labels.append(repr(l))
        else:
            display_labels.append(l)
    ax.set_yticklabels(display_labels, fontsize=7)
    ax.invert_yaxis()
    ax.set_xlabel("% of tokens")
    ax.set_title(f"Expert {e} (n={total})")

for idx in range(n_experts, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle(
    f"Layer {demo_layer}: Top BPE Tokens per Expert (WikiText-2)", fontsize=13, y=1.02
)
plt.tight_layout()
plt.show()

# ── Overall: which expert handles which token type? ──
print("Token type analysis (across all layers):")
token_categories = {
    "whitespace": {" ", "\n", "\t", "\r"},
    "punctuation": {".", ",", "!", "?", ":", ";", "'", '"', "-", "(", ")", "[", "]"},
}
for cat_name, cat_chars in token_categories.items():
    expert_counts = defaultdict(int)
    for layer_idx in sorted_layers:
        d = routing_data[layer_idx]
        idx = d["indices"][0]
        for pos in range(idx.size(0)):
            token_text = token_texts[pos]
            if any(c in token_text for c in cat_chars):
                for slot in range(idx.size(-1)):
                    expert_counts[idx[pos, slot].item()] += 1
    if expert_counts:
        best = max(expert_counts, key=expert_counts.get)
        total = sum(expert_counts.values())
        print(
            f"  {cat_name:15s} → mostly Expert {best} "
            f"({expert_counts[best]}/{total} = {expert_counts[best] / total * 100:.0f}%)"
        )

---
## Summary

### What to look for

| Section | Observation | Interpretation |
|---|---|---|
| **3. Heatmap** | Uniform column color | ✅ Load balancing working |
| | One column much darker | ❌ Expert collapsing — increase the aux loss coefficient in the training loop |
| **4. Co-activation** | Frequent pairs found | ❓ Router learns complementary expert roles |
| | All pairs equally likely | ❓ Router not structuring expert usage |
| **5. Entropy** | Mean H near 0 (confident) | ✅ Router has clear preferences |
| | Mean H near ln2 (uncertain) | ❌ Router can't distinguish tokens |
| **6. Trend** | U-shape across layers | ✅ First & last layers uncertain, middle confident (normal) |
| | Flat line across layers | ❓ Router behaves identically at every depth |
| **7. Top tokens** | Experts prefer different tokens | ✅ Semantic specialisation learned |
| | All experts have same distribution | ❌ Experts aren't differentiating |

### Key Metrics

| Metric | Ideal | Formula |
|---|---|---|
| **Load CV** | 0 (perfectly uniform) | $\sigma / \mu$ of expert load fractions |
| **Co-activation** | Structured pairs | co-occurrence matrix |
| **Mean routing entropy** | $\approx 0$ (confident) | $\frac{1}{T}\sum_t -\sum_i w_{t,i}\ln w_{t,i}$ |
| **Aux loss** | $\approx 1$ (uniform $f_i, P_i$) | $n \cdot \sum_i f_i P_i$ |